In [0]:
# ==========================================
# GOLD PLAYER STATS - SOURCE INSPECTION
# ==========================================

scorers_df = spark.table(
    "workspace.default.silver_scorers"
)

teams_df = spark.table(
    "workspace.default.silver_teams"
)

competition_df = spark.table(
    "workspace.default.silver_competition"
)

print("Scorers:", scorers_df.count())
print("Teams:", teams_df.count())

scorers_df.printSchema()

display(scorers_df)


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

scorers_df.select(
    spark_sum(col("goals").isNull().cast("int")).alias("missing_goals"),
    spark_sum(col("assists").isNull().cast("int")).alias("missing_assists"),
    spark_sum(col("matches_played").isNull().cast("int")).alias("missing_matches")
).show()


In [0]:
from pyspark.sql.functions import col, coalesce, lit, when, round

# ==========================================
# 3. CLEAN AND JOIN PLAYER DATA
# ==========================================

# Record which assists were originally missing
clean_scorers_df = scorers_df.withColumn(
    "assists_missing",
    col("assists").isNull()
).withColumn(
    "assists",
    coalesce(col("assists"), lit(0))
)

# Join player records with team details
joined_df = clean_scorers_df.alias("p").join(
    teams_df.select(
        "team_id", "stadium", "short_name"
    ).alias("t"),
    "team_id",
    "left"
)

# Join with the current competition season
joined_df = joined_df.join(
    competition_df.select(
        "season_id", "competition_name"
    ),
    "season_id",
    "inner"
)

print("Rows after joining:", joined_df.count())
display(joined_df)


In [0]:
# ==========================================
# 4. CREATE GOLD PLAYER STATISTICS
# ==========================================

gold_df = (
    joined_df
    .withColumn(
        "goal_contributions",
        col("goals") + col("assists")
    )
    .withColumn(
        "goals_per_match",
        when(
            col("matches_played") > 0,
            round(col("goals") / col("matches_played"), 2)
        )
    )
    .withColumn(
        "assists_per_match",
        when(
            col("matches_played") > 0,
            round(col("assists") / col("matches_played"), 2)
        )
    )
    .withColumn(
        "contributions_per_match",
        when(
            col("matches_played") > 0,
            round(
                col("goal_contributions") / col("matches_played"),
                2
            )
        )
    )
    .select(
        "season_id",
        "competition_name",
        "player_id",
        "player_name",
        "nationality",
        "position",
        "team_id",
        "team_name",
        "short_name",
        "stadium",
        "matches_played",
        "goals",
        "assists",
        "assists_missing",
        "penalties",
        "goal_contributions",
        "goals_per_match",
        "assists_per_match",
        "contributions_per_match"
    )
)

display(
    gold_df.orderBy(
        col("goal_contributions").desc()
    )
)


In [0]:
# ==========================================
# 5. VALIDATE GOLD PLAYER STATISTICS
# ==========================================

total_players = gold_df.count()

unique_players = gold_df.select(
    "season_id", "player_id"
).distinct().count()

# Check record counts
assert total_players == scorers_df.count(), \
    "Player count changed during joins!"

# Check for duplicates
assert total_players == unique_players, \
    "Duplicate player-season records detected!"

# Check missing team information
assert gold_df.filter(
    col("team_name").isNull()
).count() == 0, "Missing team names!"

# Validate calculated goal contributions
invalid_contributions = gold_df.filter(
    col("goal_contributions") !=
    (col("goals") + col("assists"))
).count()

assert invalid_contributions == 0, \
    "Incorrect goal contribution calculations!"

# Check for invalid negative statistics
invalid_stats = gold_df.filter(
    (col("goals") < 0) |
    (col("assists") < 0) |
    (col("matches_played") < 0)
).count()

assert invalid_stats == 0, \
    "Negative player statistics detected!"

print("Gold player statistics validation passed!")
print("Total players:", total_players)
print(
    "Players with originally missing assists:",
    gold_df.filter(col("assists_missing")).count()
)


In [0]:
# ==========================================
# 6. SAVE GOLD PLAYER STATISTICS
# ==========================================

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_player_stats")
)

print("Gold player statistics table created!")


In [0]:
%sql
SELECT
    player_name,
    team_name,
    matches_played,
    goals,
    assists,
    goal_contributions,
    goals_per_match,
    assists_per_match,
    contributions_per_match,
    assists_missing
FROM workspace.default.gold_player_stats
ORDER BY goal_contributions DESC, goals DESC;